# 隐私确认闸门 · v2：接入真实行情数据

在上一版的基础上，只改一件事：**Agent A 不再用假信号，而是用 AKShare 拉真实行情，
自己算一个 5日/20日均线金叉信号**。确认闸门那部分完全不动——这也是这种"插件化"
设计的好处：换数据源不需要碰主流程。

> 如果你在一个不能连外网的环境里跑这个 notebook（比如某些云沙箱），
> AKShare 请求会失败，代码会自动降级用示例数据，并打印一行提示。
> 在你自己电脑上跑，只要能访问东方财富网，就会是真实数据。

In [2]:
from typing import TypedDict, Literal, Optional
from uuid import uuid4
from datetime import datetime, timedelta

import akshare as ak
import pandas as pd

from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

/Users/steven/miniconda3/envs/pytorch/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## 1. 状态（和 v1 一样，没有变化）

In [3]:
class State(TypedDict, total=False):
    agent_a_result: dict
    desensitized_list: list[dict]
    user_decision: Literal["approved", "rejected"]
    agent_b_result: dict
    fusion_result: dict

## 2. 真实数据函数：拉最近 90 天行情，算 MA5/MA20 金叉死叉

拉不到数据（没有外网 / 接口变了 / 代码写错）就走 `except`，返回一个标了"示例"的信号，
notebook 不会因为网络问题跑不下去。

In [4]:
def fetch_ma_signal(ticker: str) -> str:
    try:
        end = datetime.now().strftime("%Y%m%d")
        start = (datetime.now() - timedelta(days=90)).strftime("%Y%m%d")
        df = ak.stock_zh_a_hist(symbol=ticker, period="daily",
                                 start_date=start, end_date=end, adjust="qfq")
        df["MA5"] = df["收盘"].rolling(5).mean()
        df["MA20"] = df["收盘"].rolling(20).mean()
        prev, last = df.iloc[-2], df.iloc[-1]
        if prev["MA5"] < prev["MA20"] and last["MA5"] >= last["MA20"]:
            return "MA5/MA20 金叉"
        elif prev["MA5"] > prev["MA20"] and last["MA5"] <= last["MA20"]:
            return "MA5/MA20 死叉"
        return "MA5/MA20 无交叉"
    except Exception as e:
        print(f"⚠️  拉取 {ticker} 真实行情失败（{type(e).__name__}），改用示例信号演示流程")
        return "示例信号：MA5/MA20 金叉（模拟）"


# 先试一下这个函数本身，跟图逻辑无关，方便你单独调试
fetch_ma_signal("600570")

'MA5/MA20 无交叉'

## 3. Agent A：持仓 + 真实（或降级）信号 → 生成脱敏清单

持仓列表先手写在这里，后面要接真实持仓库的话，只需要替换这一个变量的来源。

In [5]:
HOLDINGS = [
    {"ticker": "600570", "name": "恒生电子", "qty": 1000, "cost": 32.5, "industry_tags": ["金融科技"]},
]

def agent_a_node(state: State) -> dict:
    full_result = []
    for h in HOLDINGS:
        item = dict(h)
        item["signal"] = fetch_ma_signal(h["ticker"])
        full_result.append(item)

    ALLOWED = {"ticker", "name", "industry_tags"}
    desensitized = [{k: v for k, v in item.items() if k in ALLOWED} for item in full_result]
    return {"agent_a_result": full_result, "desensitized_list": desensitized}

## 4. 确认节点（和 v1 完全一样）

In [6]:
def human_confirm_node(state: State) -> dict:
    decision = interrupt({
        "will_send": state["desensitized_list"],
        "will_not_send": ["持仓数量", "成本价"],
    })
    return {"user_decision": decision["action"]}

## 5. 云端 Agent B + 融合（和 v1 完全一样，还是假数据）

下一版再把这里换成真实检索。

In [7]:
def route_after_confirm(state: State) -> str:
    return "agent_b" if state["user_decision"] == "approved" else END

def agent_b_node(state: State) -> dict:
    tickers = [item["ticker"] for item in state["desensitized_list"]]
    return {"agent_b_result": {"summary": f"围绕 {tickers} 检索到公开新闻，情绪偏正面。"}}

def fusion_node(state: State) -> dict:
    # 融合时可以把本地真实信号也带进理由里
    signals = [h.get("signal") for h in state["agent_a_result"]]
    return {"fusion_result": {
        "advice": "持有并关注",
        "local_signal": signals,
        "cloud_info": state["agent_b_result"],
    }}

## 6. 组图（和 v1 完全一样）

In [8]:
graph = StateGraph(State)
graph.add_node("agent_a", agent_a_node)
graph.add_node("human_confirm", human_confirm_node)
graph.add_node("agent_b", agent_b_node)
graph.add_node("fusion", fusion_node)

graph.add_edge(START, "agent_a")
graph.add_edge("agent_a", "human_confirm")
graph.add_conditional_edges("human_confirm", route_after_confirm, {"agent_b": "agent_b", END: END})
graph.add_edge("agent_b", "fusion")
graph.add_edge("fusion", END)

app = graph.compile(checkpointer=MemorySaver())

## 7. 跑起来

这一步会真的尝试联网拉行情。拉到真实数据，`will_send` 里的持仓还是只有代码/名称/行业标签——
但融合结果里的 `local_signal` 会是基于真实均线算出来的，不再是编的。

In [9]:
thread_id = str(uuid4())
config = {"configurable": {"thread_id": thread_id}}

result = app.invoke({}, config=config)
result["__interrupt__"][0].value

{'will_send': [{'ticker': '600570',
   'name': '恒生电子',
   'industry_tags': ['金融科技']}],
 'will_not_send': ['持仓数量', '成本价']}

## 8. 确认，看最终结果

In [10]:
final = app.invoke(Command(resume={"action": "approved"}), config=config)
final["fusion_result"]

{'advice': '持有并关注',
 'local_signal': ['MA5/MA20 无交叉'],
 'cloud_info': {'summary': "围绕 ['600570'] 检索到公开新闻，情绪偏正面。"}}

---
### 下一步候选（你挑一个，我们接着加）

- Agent B 换成真实的 Web 检索（而不是编一句话）
- 白名单从"一行过滤"升级成严格 schema 校验（多传字段直接报错）
- HOLDINGS 从手写列表换成读一个本地文件/小数据库
- 让用户在确认时能删掉某个标的（`edited_list`）
